In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

from Fitter import post_correct_update_matrix

# =========================
# Paths and settings
# =========================
data_path = r"D:\sc_modelling-amir-2025\preprocessed_Data_mouse_March2025.csv"

output_dir = r"D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse"
os.makedirs(output_dir, exist_ok=True)

participants = ['QP0100', 'QP0101', 'QP0103', 'QP0121', 'QP062', 'QP063', 'QP070', 'QP071']

# =========================
# Colormap and normalization
# =========================
colors = ["orange", "white", "purple"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

norm_update = TwoSlopeNorm(vmin=-0.2, vcenter=0, vmax=0.2)
norm_cond = TwoSlopeNorm(vmin=0.0, vcenter=0.5, vmax=1.0)

extent = [-1, 1, -1, 1]
tick_positions = np.linspace(-1, 1, 9)
tick_labels = [-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1]


# =========================
# Binned-count empirical method
# =========================
def post_correct_binned_update_matrix(s, chooseB, reward, No_response, Not_Blockstart, n_bins=8):
    s = np.asarray(s)
    chooseB = np.asarray(chooseB)
    reward = np.asarray(reward)
    No_response = np.asarray(No_response)
    Not_Blockstart = np.asarray(Not_Blockstart)

    edges = np.linspace(-1, 1, n_bins + 1)

    current_bin = np.digitize(s[1:], edges) - 1
    previous_bin = np.digitize(s[:-1], edges) - 1

    valid = (
        (reward[:-1] == 1) &
        (No_response[:-1] == False) &
        (No_response[1:] == False) &
        (Not_Blockstart[1:] == True) &
        (current_bin >= 0) & (current_bin < n_bins) &
        (previous_bin >= 0) & (previous_bin < n_bins)
    )

    current_bin = current_bin[valid]
    previous_bin = previous_bin[valid]
    choice_current = chooseB[1:][valid]

    N = np.zeros((n_bins, n_bins))
    B = np.zeros((n_bins, n_bins))

    for i, j, ch in zip(current_bin, previous_bin, choice_current):
        N[i, j] += 1
        B[i, j] += ch

    conditional_matrix = np.full((n_bins, n_bins), np.nan)
    mask = N > 0
    conditional_matrix[mask] = B[mask] / N[mask]

    N_total = np.zeros(n_bins)
    B_total = np.zeros(n_bins)

    for i, ch in zip(current_bin, choice_current):
        N_total[i] += 1
        B_total[i] += ch

    total_psychometric = np.full(n_bins, np.nan)
    row_mask = N_total > 0
    total_psychometric[row_mask] = B_total[row_mask] / N_total[row_mask]

    update_matrix = conditional_matrix - total_psychometric[:, None]

    return update_matrix, conditional_matrix, N, B, total_psychometric


# =========================
# Helpers
# =========================
def style_axis(ax, row_idx, col_idx, title):
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=8)

    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels, fontsize=8)

    ax.set_title(title, fontsize=10)

    if col_idx != 0:
        ax.set_yticklabels([])

    if row_idx == 0:
        ax.set_xticklabels([])


def plot_empirical_grid(matrices, figure_title, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(9, 8), dpi=300, constrained_layout=True)

    plot_items = [
        ("fit_update", "Fitted update", norm_update),
        ("bin_update", "Binned update", norm_update),
        ("fit_cond", "Fitted conditional", norm_cond),
        ("bin_cond", "Binned conditional", norm_cond),
    ]

    last_im_update = None
    last_im_cond = None

    for idx, (key, title, norm) in enumerate(plot_items):
        row_idx = idx // 2
        col_idx = idx % 2
        ax = axes[row_idx, col_idx]

        mat = matrices[key]

        im = ax.imshow(
            mat,
            cmap=cmap,
            norm=norm,
            aspect="equal",
            interpolation="nearest",
            origin="upper",
            extent=extent
        )

        if "update" in key:
            last_im_update = im
        else:
            last_im_cond = im

        style_axis(ax, row_idx, col_idx, title)

    fig.supxlabel("Previous stimulus", fontsize=13)
    fig.supylabel("Current stimulus", fontsize=13)

    cbar1 = fig.colorbar(last_im_update, ax=axes[:, 0], shrink=0.75)
    cbar1.set_label("Update strength", fontsize=10)

    cbar2 = fig.colorbar(last_im_cond, ax=axes[:, 1], shrink=0.75)
    cbar2.set_label("P(choose B)", fontsize=10)

    fig.suptitle(figure_title, fontsize=15)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved plot: {save_path}")


def get_empirical_matrices(pid_df):
    pid_df = pid_df.copy().reset_index(drop=True)
    pid_df["is_not_start_of_block"] = pid_df["block"].eq(pid_df["block"].shift())

    s = pid_df["stim_relative"].to_numpy()
    chooseB = pid_df["choice"].to_numpy()
    rewards = pid_df["correct"].to_numpy()
    No_response = pid_df["No_response"].to_numpy()
    Not_Blockstart = pid_df["is_not_start_of_block"].to_numpy()

    fit_update, fit_cond = post_correct_update_matrix(
        s, chooseB, rewards, No_response, Not_Blockstart
    )

    bin_update, bin_cond, N, B, total_bin = post_correct_binned_update_matrix(
        s, chooseB, rewards, No_response, Not_Blockstart
    )

    return {
        "fit_update": fit_update[::-1],
        "fit_cond": fit_cond[::-1],
        "bin_update": bin_update[::-1],
        "bin_cond": bin_cond[::-1],
        "N": N[::-1],
        "B": B[::-1],
        "total_bin": total_bin[::-1],
    }


# =========================
# Load data
# =========================
df = pd.read_csv(data_path)

# =========================
# Per-participant plots
# =========================
all_matrices = []

for participant in participants:
    pid_df = df[df["Participant_ID"] == participant].reset_index(drop=True)

    if len(pid_df) == 0:
        print(f"No data found for participant {participant}")
        continue

    matrices = get_empirical_matrices(pid_df)
    all_matrices.append(matrices)

    save_path = os.path.join(output_dir, f"{participant}_empirical_matrices.png")

    plot_empirical_grid(
        matrices=matrices,
        figure_title=f"Participant {participant}",
        save_path=save_path
    )


# =========================
# Mean-over-participants plot
# =========================
mean_matrices = {}

for key in ["fit_update", "fit_cond", "bin_update", "bin_cond"]:
    mats = [m[key] for m in all_matrices]
    mean_matrices[key] = np.nanmean(np.stack(mats, axis=0), axis=0)

mean_save_path = os.path.join(output_dir, "MEAN_empirical_matrices.png")

plot_empirical_grid(
    matrices=mean_matrices,
    figure_title="Mean over participants",
    save_path=mean_save_path
)

print("Done.")

C:\Users\Akrami_Lab5\AppData\Local\Temp\ipykernel_6596\2689700051.py:192: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP0100_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP0101_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP0103_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP0121_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP062_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP063_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP070_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\QP071_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_mouse\MEAN_empirical_matrices.png
Done.


In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

from Fitter import post_correct_update_matrix

# =========================
# Paths and settings
# =========================
data_path = r"D:\sc_modelling-amir-2025\Human_Data_switching.csv"

output_dir = r"D:\sc_modelling-amir-2025\Empirical_matrix_plots_human"
os.makedirs(output_dir, exist_ok=True)

pid_col = "Participant Private ID"

participants = [
    '6363339', '6363334', '6363353', '6363338', '6363346', '6363343',
    '6363329', '6363354', '6363347', '6363328', '6363330', '6363341',
    '6363407', '6363430', '6363368', '6363349', '6363364', '6269773',
    '6269770', '6269757', '6269881', '6357507', '6357495', '6357501',
    '6357497', '6357556', '6357698', '6357829'
]
participants = [str(p) for p in participants]

# =========================
# Colormap and normalization
# =========================
colors = ["orange", "white", "purple"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

norm_update = TwoSlopeNorm(vmin=-0.2, vcenter=0, vmax=0.2)
norm_cond = TwoSlopeNorm(vmin=0.0, vcenter=0.5, vmax=1.0)

extent = [-1, 1, -1, 1]
tick_positions = np.linspace(-1, 1, 9)
tick_labels = [-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1]


# =========================
# Binned-count empirical method
# =========================
def post_correct_binned_update_matrix(s, chooseB, reward, No_response, Not_Blockstart, n_bins=8):
    s = np.asarray(s)
    chooseB = np.asarray(chooseB)
    reward = np.asarray(reward)
    No_response = np.asarray(No_response)
    Not_Blockstart = np.asarray(Not_Blockstart)

    edges = np.linspace(-1, 1, n_bins + 1)

    current_bin = np.digitize(s[1:], edges) - 1
    previous_bin = np.digitize(s[:-1], edges) - 1

    valid = (
        (reward[:-1] == 1) &
        (No_response[:-1] == False) &
        (No_response[1:] == False) &
        (Not_Blockstart[1:] == True) &
        (current_bin >= 0) & (current_bin < n_bins) &
        (previous_bin >= 0) & (previous_bin < n_bins)
    )

    current_bin = current_bin[valid]
    previous_bin = previous_bin[valid]
    choice_current = chooseB[1:][valid]

    N = np.zeros((n_bins, n_bins))
    B = np.zeros((n_bins, n_bins))

    for i, j, ch in zip(current_bin, previous_bin, choice_current):
        N[i, j] += 1
        B[i, j] += ch

    conditional_matrix = np.full((n_bins, n_bins), np.nan)
    mask = N > 0
    conditional_matrix[mask] = B[mask] / N[mask]

    N_total = np.zeros(n_bins)
    B_total = np.zeros(n_bins)

    for i, ch in zip(current_bin, choice_current):
        N_total[i] += 1
        B_total[i] += ch

    total_psychometric = np.full(n_bins, np.nan)
    row_mask = N_total > 0
    total_psychometric[row_mask] = B_total[row_mask] / N_total[row_mask]

    update_matrix = conditional_matrix - total_psychometric[:, None]

    return update_matrix, conditional_matrix, N, B, total_psychometric


# =========================
# Helpers
# =========================
def style_axis(ax, row_idx, col_idx, title):
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=8)

    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels, fontsize=8)

    ax.set_title(title, fontsize=10)

    if col_idx != 0:
        ax.set_yticklabels([])

    if row_idx == 0:
        ax.set_xticklabels([])


def plot_empirical_grid(matrices, figure_title, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(9, 8), dpi=300, constrained_layout=True)

    plot_items = [
        ("fit_update", "Fitted update", norm_update),
        ("bin_update", "Binned update", norm_update),
        ("fit_cond", "Fitted conditional", norm_cond),
        ("bin_cond", "Binned conditional", norm_cond),
    ]

    last_im_update = None
    last_im_cond = None

    for idx, (key, title, norm) in enumerate(plot_items):
        row_idx = idx // 2
        col_idx = idx % 2
        ax = axes[row_idx, col_idx]

        mat = matrices[key]

        im = ax.imshow(
            mat,
            cmap=cmap,
            norm=norm,
            aspect="equal",
            interpolation="nearest",
            origin="upper",
            extent=extent
        )

        if "update" in key:
            last_im_update = im
        else:
            last_im_cond = im

        style_axis(ax, row_idx, col_idx, title)

    fig.supxlabel("Previous stimulus", fontsize=13)
    fig.supylabel("Current stimulus", fontsize=13)

    cbar1 = fig.colorbar(last_im_update, ax=axes[:, 0], shrink=0.75)
    cbar1.set_label("Update strength", fontsize=10)

    cbar2 = fig.colorbar(last_im_cond, ax=axes[:, 1], shrink=0.75)
    cbar2.set_label("P(choose B)", fontsize=10)

    fig.suptitle(figure_title, fontsize=15)

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved plot: {save_path}")


def get_empirical_matrices(pid_df):
    pid_df = pid_df.copy().reset_index(drop=True)
    pid_df["is_not_start_of_block"] = pid_df["block"].eq(pid_df["block"].shift())

    s = pid_df["stim_relative"].to_numpy()
    chooseB = pid_df["choice"].to_numpy()
    rewards = pid_df["correct"].to_numpy()
    No_response = pid_df["No_response"].to_numpy()
    Not_Blockstart = pid_df["is_not_start_of_block"].to_numpy()

    fit_update, fit_cond = post_correct_update_matrix(
        s, chooseB, rewards, No_response, Not_Blockstart
    )

    bin_update, bin_cond, N, B, total_bin = post_correct_binned_update_matrix(
        s, chooseB, rewards, No_response, Not_Blockstart
    )

    return {
        "fit_update": fit_update[::-1],
        "fit_cond": fit_cond[::-1],
        "bin_update": bin_update[::-1],
        "bin_cond": bin_cond[::-1],
        "N": N[::-1],
        "B": B[::-1],
        "total_bin": total_bin[::-1],
    }


# =========================
# Load data
# =========================
df = pd.read_csv(data_path, low_memory=False)
df['No_response']=False

if pid_col not in df.columns:
    raise ValueError(f"Column '{pid_col}' not found. Available columns:\n{df.columns.tolist()}")

# Convert IDs safely: handles 6363339, 6363339.0, and mixed types
df[pid_col] = (
    pd.to_numeric(df[pid_col], errors="coerce")
    .astype("Int64")
    .astype(str)
)

print("Using participant column:", pid_col)
print("Number of rows:", len(df))
print("Number of unique participants:", df[pid_col].nunique())
print("First unique participant IDs:")
print(sorted(df[pid_col].dropna().unique())[:50])

# =========================
# Per-participant plots
# =========================
all_matrices = []

for participant in participants:
    pid_df = df[df[pid_col] == participant].reset_index(drop=True)

    if len(pid_df) == 0:
        print(f"No data found for participant {participant}")
        continue

    matrices = get_empirical_matrices(pid_df)
    all_matrices.append(matrices)

    save_path = os.path.join(output_dir, f"{participant}_empirical_matrices.png")

    plot_empirical_grid(
        matrices=matrices,
        figure_title=f"Participant {participant}",
        save_path=save_path
    )


# =========================
# Mean-over-participants plot
# =========================
if len(all_matrices) == 0:
    print("No participant matrices were computed. Check participant IDs and column names.")
else:
    mean_matrices = {}

    for key in ["fit_update", "fit_cond", "bin_update", "bin_cond"]:
        mats = [m[key] for m in all_matrices]
        mean_matrices[key] = np.nanmean(np.stack(mats, axis=0), axis=0)

    mean_save_path = os.path.join(output_dir, "MEAN_empirical_matrices.png")

    plot_empirical_grid(
        matrices=mean_matrices,
        figure_title="Mean over participants",
        save_path=mean_save_path
    )

print("Done.")

Using participant column: Participant Private ID
Number of rows: 46748
Number of unique participants: 39
First unique participant IDs:
['6269757', '6269770', '6269773', '6269881', '6357495', '6357497', '6357499', '6357501', '6357507', '6357542', '6357556', '6357698', '6357829', '6363327', '6363328', '6363329', '6363330', '6363332', '6363334', '6363335', '6363336', '6363337', '6363338', '6363339', '6363341', '6363343', '6363345', '6363346', '6363347', '6363349', '6363350', '6363353', '6363354', '6363364', '6363368', '6363399', '6363401', '6363407', '6363430']
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_human\6363339_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_human\6363334_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_human\6363353_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empirical_matrix_plots_human\6363338_empirical_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Empir